# AEGIS AI — Kill-Feed Detector Training (RF-DETR Nano) 🎯

**Ye notebook kya karta hai (simple shabdon me):**
BGMI broadcast ke frame me **kill-feed** (upar-right corner me aane wala "PlayerA ne PlayerB ko mara" wala text box) ko pehchanne wala AI model train karta hai. Ye model baad me AEGIS ke vision pipeline me lagega.

**Dataset:** 100,000 frames, **10 chunk zips** me aayega (~800MB each: images + labels). Notebook saare chunks **khud** unzip karke ek dataset bana lega — aapko alag se kuch nahi karna. Dataset me `train/` + `val/` ke saath **`test/` split** bhi hai (500-1000 clean frames, ek bilkul alag tournament se) — final report card usi par banega.

**Aapko kya karna hai — sirf 4 steps:**
1. **GPU on karo:** Kaggle notebook ke right side me *Settings* → *Accelerator* → **GPU T4** select karo.
2. **Dataset jodho:** Right side me *+ Add data* → *Your Datasets* → apna `aegis-killfeed-dataset` add karo. (Chunks upload karne ka tarika `docs/KAGGLE_TRAINING.md` me hai.)
3. Neeche **Cell 3** me `DATASET_SLUG` me apna Kaggle username likho.
4. Upar **Run** (▶▶) dabao aur chai pi lo ☕ — 100,000 images × 50 epochs me **kai ghante** lagenge.

**Zaroori note:** Ye notebook **Kaggle** par chalane ke liye bana hai (free GPU: hafte me ~30 ghante). Google Colab wala purana notebook hata diya gaya hai.


In [ ]:
# Cell 1 — RF-DETR install karo (sirf pehli baar; ~2-4 minute lag sakte hain)
!pip install -q rfdetr "rfdetr[onnxexport]"
print("Install complete ✔  (agar neeche import me error aaye to: Kernel → Restart karke dobara chalao)")


## Step 1 — GPU check 🔌

Neeche wala cell chalao. Agar `CUDA available: True` aur **Tesla T4** dikhe to sab sahi hai. Agar `False` aaye to Settings → Accelerator → **GPU** select karo.


In [ ]:
# Cell 2 — GPU hai ya nahi?
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU naam:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
else:
    print("⚠️ GPU NAHI mila! Right side Settings → Accelerator → GPU T4 select karo, phir dobara chalao.")


## Step 2 — Dataset jodho 📦

`DATASET_SLUG` me apna Kaggle **username** likho. Example: agar aapka username `nishant07` hai aur dataset ka naam `aegis-killfeed-dataset` hai, to likho `"nishant07/aegis-killfeed-dataset"`.


In [ ]:
# Cell 3 — Apna dataset slug yahan likho (sirf username badalna hai)
DATASET_SLUG = "vivekyadav54/aegis-killfeed-dataset"

from pathlib import Path
INPUT = Path("/kaggle/input")

# Kaggle zip files ko auto-extract kar deta hai — dono case handle karo
zips = sorted(INPUT.rglob("*.zip"))
print(f"Kul {len(zips)} zip chunks mile /kaggle/input me:")
for z in zips:
    print(f"  ✔ {z.relative_to('/kaggle/input')} ({z.stat().st_size/1e6:.0f} MB)")

# Agar zip nahi mile to extracted files check karo
if not zips:
    extracted_imgs = list(INPUT.rglob("*.jpg")) + list(INPUT.rglob("*.jpeg")) + list(INPUT.rglob("*.png"))
    print(f"\n📂 Extracted files mile: {len(extracted_imgs)} images")
    if extracted_imgs:
        print(f"  ✔ Dataset pehle se extracted hai — zip ki zaroorat nahi!")
        print(f"  ✔ Example: {extracted_imgs[0].relative_to(INPUT)}")
    else:
        print("⚠️ Koi zip nahi mila! Dataset attach kiya? Right side + Add data → Your Datasets")


In [ ]:
# Cell 3b — Dataset taiyaar karo (zip ya extracted, dono chalega)
# Kaggle zip files ko auto-extract kar deta hai, isliye dono case handle karte hain.
import zipfile, shutil, random, yaml
from pathlib import Path

INPUT = Path("/kaggle/input")
DS = Path("/kaggle/working/aegis_ds")  # final dataset: images/train, images/val, labels/train, labels/val
if DS.exists():
    shutil.rmtree(DS)

CLASS_NAMES = ["killfeed"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
random.seed(42)
counts = {"train": 0, "val": 0, "test": 0}

# --- Step 1: Source dhoondo (zip ya extracted) ---
zips = sorted(INPUT.rglob("*.zip"))

if zips:
    print(f"📦 {len(zips)} zip chunks mile — unzip kar raha hun...")
    # Zip se nikalke tmp me rakho, phir process karo
    sources = []  # (label, tmp_path) pairs
    for ci, z in enumerate(zips):
        tmp = Path(f"/tmp/chunk_{ci:02d}")
        if tmp.exists():
            shutil.rmtree(tmp)
        tmp.mkdir(parents=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(tmp)
        sources.append((z.name, tmp))
else:
    # Kaggle ne pehle se extract kar diya — INPUT ko hi source banao
    extracted = [p for p in INPUT.rglob("*") if p.suffix.lower() in IMG_EXTS]
    assert extracted, "Koi zip ya image nahi mili — dataset attach karo (+ Add data → Your Datasets)"
    print(f"📂 {len(extracted)} extracted images mile — direct use kar raha hun...")
    sources = [("extracted", INPUT)]

# --- Step 2: dataset.yaml se class names lo (pehle source se) ---
for y in sources[0][1].rglob("dataset.yaml"):
    n = yaml.safe_load(open(y)).get("names", ["killfeed"])
    CLASS_NAMES = n if isinstance(n, list) else [n[k] for k in sorted(n)]
    break

# --- Step 3: Har source se images nikalo aur DS me copy karo ---
for ci, (label, src) in enumerate(sources):
    lbl_by_stem = {}
    for t in src.rglob("*.txt"):
        # dataset.yaml ko label mat samjho
        if t.name == "dataset.yaml":
            continue
        lbl_by_stem.setdefault(t.stem, t)

    imgs = sorted(p for p in src.rglob("*") if p.suffix.lower() in IMG_EXTS)

    for img in imgs:
        parts = [p.lower() for p in img.parts]
        if "train" in parts:
            split = "train"
        elif "test" in parts:
            split = "test"  # alag tournament ke clean frames — training me KABHI use nahi honge
        elif "val" in parts or "valid" in parts:
            split = "val"
        else:
            split = "val" if random.random() < 0.10 else "train"  # 90/10 split

        dest_img = DS / "images" / split / f"chunk{ci:02d}_{img.name}"
        dest_img.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(img, dest_img)

        lbl = lbl_by_stem.get(img.stem)
        if lbl is not None:
            dest_lbl = DS / "labels" / split / (dest_img.stem + ".txt")
            dest_lbl.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(lbl, dest_lbl)

        counts[split] += 1

    # Zip wale tmp folders saaf karo (extracted wala INPUT hai, use mat udao!)
    if label != "extracted":
        shutil.rmtree(src)

    print(f"✔ {label}: {len(imgs)} images")

# --- Step 4: Final dataset.yaml banao ---
print("Classes:", CLASS_NAMES)
yaml.safe_dump({"path": str(DS), "train": "images/train", "val": "images/val",
                "test": "images/test", "names": CLASS_NAMES},
               open(DS / "dataset.yaml", "w"))
print(f"Saaf dataset taiyaar ✔  train: {counts['train']}, val: {counts['val']}, test: {counts['test']}  → {DS}")

if counts["test"] == 0:
    print("⚠️ WARNING: test/ split nahi mila! 'Final report card' (asli duniya ka andaza) nahi ban payega.")
    print("   Dataset me 'test' naam ke folder me 500-1000 clean frames (ek alag tournament se) jodho.")
else:
    print(f"✔ test/ me {counts['test']} clean frames — ye training me kabhi use nahi honge (final report card).")


## Step 3 — YOLO format → COCO format badlo 🔄

Hamara dataset **YOLO format** me hai (`.txt` label files), lekin RF-DETR training ke liye **COCO format** (`.json` annotation file) mangta hai. Neeche wala cell ye conversion **automatic** karta hai — aapko kuch nahi karna.

- `images/val` → `valid` split banega (RF-DETR `val` ko `valid` kehta hai)
- `images/test` → `test` split banega — **ASLI test split** (alag tournament ke 500-1000 clean frames, training me kabhi nahi). Agar test/ na mile to val me se har 10vi image ka kaamchalau test banega (lekin asli test/ jodna behtar hai)
- Images copy hongi `/kaggle/working/coco_dataset/` me; conversion ke baad purana YOLO folder delete ho jayega (disk bachane ke liye)


In [ ]:
# Cell 4 — YOLO (.txt) → COCO (.json) conversion (automatic)
# RF-DETR ko COCO format chahiye; hamara saaf dataset (Cell 3b se) YOLO format me hai.
import json, shutil
from pathlib import Path
from PIL import Image

print("Class names:", CLASS_NAMES)  # Cell 3b se aaya
COCO = Path("/kaggle/working/coco_dataset")

def yolo_split_to_coco(img_dir, lbl_dir, out_dir):
    img_dir, lbl_dir, out_dir = Path(img_dir), Path(lbl_dir), Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    images, annotations = [], []
    ann_id, img_id = 1, 1
    for img_path in sorted(img_dir.iterdir()):
        if img_path.suffix.lower() not in (".jpg", ".jpeg", ".png", ".webp"):
            continue
        with Image.open(img_path) as im:
            w, h = im.size
        shutil.copy2(img_path, out_dir / img_path.name)
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().splitlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls = int(float(parts[0]))
                cx, cy, bw, bh = map(float, parts[1:5])
                x = max(0.0, (cx - bw / 2) * w)
                y = max(0.0, (cy - bh / 2) * h)
                bw_px = min(bw * w, w - x)
                bh_px = min(bh * h, h - y)
                annotations.append({"id": ann_id, "image_id": img_id, "category_id": cls + 1,
                                    "bbox": [x, y, bw_px, bh_px], "area": bw_px * bh_px, "iscrowd": 0})
                ann_id += 1
        images.append({"id": img_id, "file_name": img_path.name, "width": w, "height": h})
        img_id += 1
    coco = {"images": images, "annotations": annotations,
            "categories": [{"id": i + 1, "name": n} for i, n in enumerate(CLASS_NAMES)]}
    (out_dir / "_annotations.coco.json").write_text(json.dumps(coco))
    return len(images), len(annotations)

# train → train
n_img, n_ann = yolo_split_to_coco(DS / "images" / "train", DS / "labels" / "train", COCO / "train")
print(f"train: {n_img} images, {n_ann} boxes")

IMG_OK = (".jpg", ".jpeg", ".png", ".webp")
def list_imgs(d):
    d = Path(d)
    return sorted(p for p in d.glob("*") if p.suffix.lower() in IMG_OK) if d.exists() else []

# val → valid
n_img, n_ann = yolo_split_to_coco(DS / "images" / "val", DS / "labels" / "val", COCO / "valid")
print(f"valid: {n_img} images, {n_ann} boxes")

# test → test  (ASLI test split: alag tournament ke clean frames — training me kabhi nahi)
test_imgs = list_imgs(DS / "images" / "test")
if test_imgs:
    n_img, n_ann = yolo_split_to_coco(DS / "images" / "test", DS / "labels" / "test", COCO / "test")
    print(f"test: {n_img} images, {n_ann} boxes  ← ASLI test split ✔ (final report card isi par banega)")
else:
    print("⚠️ Asli test/ split nahi mila — val me se har 10vi image se kaamchalau test ban raha hai.")
    val_imgs = list_imgs(DS / "images" / "val")
    test_names = {p.name for i, p in enumerate(val_imgs) if i % 10 == 0}
    tmp_img, tmp_lbl = Path("/tmp/split_img"), Path("/tmp/split_lbl")
    for d in (tmp_img, tmp_lbl):
        if d.exists():
            shutil.rmtree(d)
        d.mkdir(parents=True)
    for p in val_imgs:
        if p.name in test_names:
            shutil.copy2(p, tmp_img / p.name)
            lp = (DS / "labels" / "val") / (p.stem + ".txt")
            if lp.exists():
                shutil.copy2(lp, tmp_lbl / lp.name)
    n_img, n_ann = yolo_split_to_coco(tmp_img, tmp_lbl, COCO / "test")
    print(f"test: {n_img} images, {n_ann} boxes (kaamchalau — asli test/ jodne ki sifarish hai)")
print("COCO dataset taiyaar ✔  →", COCO)
# YOLO wala saaf dataset ab nahi chahiye (disk bachao) — COCO copy hi kaam aayegi
shutil.rmtree(DS)
print("Purana YOLO folder saaf kiya (disk bacha) ✔")


## Step 4 — Training 🏋️ (sabse lamba step)

Neeche wala cell **RF-DETR-Nano** model ko train karega:
- **50 epochs** (poora dataset 50 baar dekhega)
- `batch_size=4, grad_accum_steps=4` — ye setting khaas **Tesla T4 (16GB)** GPU ke liye hai (RF-DETR ki official guide ke hisaab se)
- Shuruaat COCO par pre-trained weights se hogi, isliye seekhne me asaani hogi

⏳ **Time (imaandaar note):** 100,000 images × 50 epochs T4 GPU par **kai ghante** lagenge. Exact time pehle se nahi bataya ja sakta — ye image size aur data loading par nirbhar hai; Run dabane ke baad jo ETA dikhe, use dekho.
- **Kaggle session limit:** GPU session **max ~9-12 ghante** chalta hai (Kaggle docs: 12 ghante; GPU tips page: 9 ghante) — uske baad session band ho jata hai. Isliye training ke **checkpoints** `/kaggle/working/rfdetr_output/` me save hote rehte hain.
- **Beech me ruka to:** Neeche training cell me `resume` wali 2 lines uncomment karo aur checkpoint ka path likho — training wahi se aage badhegi, dobara zero se nahi.
- **GPU quota:** Hafte me **~30 ghante** free GPU milta hai. Badi training shuru karne se pehle Kaggle → Settings me bacha hua quota dekh lo.
- **Epochs kam karne hon to:** `epochs=50` ko 25-30 kar sakte ho (time bachega, accuracy thodi kam aa sakti hai) — lekin **default 50 hi rakha hai**.


In [ ]:
# Cell 5 — RF-DETR-Nano training shuru karo
from rfdetr import RFDETRNano

model = RFDETRNano()  # Nano = sabse chhota/fast RF-DETR, Apache 2.0 license (commercial-safe)

model.train(
    dataset_dir="/kaggle/working/coco_dataset",
    epochs=50,           # kam time ho to 25-30 kar sakte ho (default 50)
    batch_size=4,        # T4 GPU ke liye (official guide)
    grad_accum_steps=4,  # effective batch = 4x4 = 16
    lr=1e-4,
    output_dir="/kaggle/working/rfdetr_output",  # checkpoints yahan save honge
    # Beech me ruka tha? Neeche wali line uncomment karo aur training wahi se resume hogi:
    # resume="/kaggle/working/rfdetr_output/checkpoint_best_total.pth",
)
print("Training khatm ✔  Checkpoints:", "/kaggle/working/rfdetr_output")


## Step 5 — Result numbers ka matlab samjho 📊

**Do tarah ke numbers aayenge — farak samajhna zaroori hai:**

1. **"Training ke dauraan" wale numbers (valid set par):** Training ke baad model valid images par kaisa chala. Ye *practice test* jaise hain — inhe dekhkar pata chalta hai training sahi direction me ja rahi hai ya nahi.
2. **"Final report card" wale numbers (test set par):** Sabse aakhir me, best model ko **500-1000 clean frames** par parkha jayega jo **ek bilkul alag tournament** se hain aur training me **kabhi use nahi hue**. Ye **"asli duniya ka andaza"** hai — asli broadcast par model kaisa chalega, iska sabse imaandaar measure. **Isi number par bharosa karo.**

Numbers ka matlab (dono jagah same):
- **Precision** (0 se 1 ke beech): Model ne jitne kill-feed boxes banaye, unme se kitne **sahi** the. 0.9 = 100 me se 90 sahi. Zyada = kam galat alarm.
- **Recall** (0 se 1 ke beech): Asli kill-feed boxes me se model ne kitne **pakde**. 0.8 = 100 me se 80 pakde, 20 choot gaye. Zyada = kam miss.

Dono 1 ke jitne kareeb, utna behtar. **Koi number pehle se promise nahi kiya ja sakta** — ye aapke dataset ki size aur quality par nirbhar karega. Data zyada aur saaf hoga to numbers behtar aayenge.


In [ ]:
# Cell 6 — Valid set par evaluation ("training ke dauraan" wale numbers)
import json
from pathlib import Path
from rfdetr import RFDETRNano

OUT = Path("/kaggle/working/rfdetr_output")
ckpts = sorted(OUT.glob("*.pth"))
print("Checkpoints:", [c.name for c in ckpts])
best = [c for c in ckpts if "best" in c.name.lower()]
best_ckpt = sorted(best)[-1] if best else ckpts[-1]
print("Best checkpoint istemal ho raha hai:", best_ckpt.name)

model = RFDETRNano(pretrain_weights=str(best_ckpt))

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0

coco = json.loads(Path("/kaggle/working/coco_dataset/valid/_annotations.coco.json").read_text())
gt_by_img = {}
for ann in coco["annotations"]:
    x, y, w, h = ann["bbox"]
    gt_by_img.setdefault(ann["image_id"], []).append([x, y, x + w, y + h])
img_by_id = {im["id"]: im["file_name"] for im in coco["images"]}

TP = FP = FN = 0
for img_id, gt_boxes in gt_by_img.items():
    det = model.predict(f"/kaggle/working/coco_dataset/valid/{img_by_id[img_id]}", threshold=0.5)
    order = sorted(range(len(det.xyxy)), key=lambda i: float(det.confidence[i]), reverse=True)
    matched = [False] * len(gt_boxes)
    for i in order:
        pb = det.xyxy[i].tolist()
        biou, bj = 0.0, -1
        for j, gb in enumerate(gt_boxes):
            if matched[j]:
                continue
            v = iou(pb, gb)
            if v > biou:
                biou, bj = v, j
        if biou >= 0.5:
            TP += 1
            matched[bj] = True
        else:
            FP += 1
    FN += sum(1 for m in matched if not m)

precision = TP / (TP + FP) if TP + FP else 0.0
recall = TP / (TP + FN) if TP + FN else 0.0
print(f"\nPrecision: {precision:.3f}  (kitne detections sahi the)")
print(f"Recall:    {recall:.3f}  (kitne asli kill-feed pakde gaye)")
print("\n(Ye 'training ke dauraan' wale numbers hain — final report card agle cell me.)")


## Step 5b — FINAL report card 📝 (test set: asli duniya ka andaza)

Ab best model ko **test set** par parkho — 500-1000 clean frames jo **ek bilkul alag tournament** se hain aur training me **kabhi use nahi hue**. Jo numbers yahan aayenge, wahi **asli duniya ka sabse imaandaar andaza** hai.


In [ ]:
# Cell 7 — FINAL evaluation: test set par (final report card / asli duniya ka andaza)
import json
from pathlib import Path
from rfdetr import RFDETRNano

TEST_JSON = Path("/kaggle/working/coco_dataset/test/_annotations.coco.json")
if not TEST_JSON.exists():
    print("⚠️ test/ ka COCO data nahi mila — final report card skip ho raha hai.")
    print("   (Cell 4 me test split check karo; dataset me test/ folder jodho.)")
else:
    OUT = Path("/kaggle/working/rfdetr_output")
    ckpts = sorted(OUT.glob("*.pth"))
    best = [c for c in ckpts if "best" in c.name.lower()]
    best_ckpt = sorted(best)[-1] if best else ckpts[-1]
    print("Best checkpoint:", best_ckpt.name)
    model = RFDETRNano(pretrain_weights=str(best_ckpt))

    def iou(a, b):
        ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
        ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
        iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
        inter = iw * ih
        ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
        return inter / ua if ua > 0 else 0.0

    coco = json.loads(TEST_JSON.read_text())
    gt_by_img = {}
    for ann in coco["annotations"]:
        x, y, w, h = ann["bbox"]
        gt_by_img.setdefault(ann["image_id"], []).append([x, y, x + w, y + h])
    img_by_id = {im["id"]: im["file_name"] for im in coco["images"]}

    TP = FP = FN = 0
    for img_id, gt_boxes in gt_by_img.items():
        det = model.predict(f"/kaggle/working/coco_dataset/test/{img_by_id[img_id]}", threshold=0.5)
        order = sorted(range(len(det.xyxy)), key=lambda i: float(det.confidence[i]), reverse=True)
        matched = [False] * len(gt_boxes)
        for i in order:
            pb = det.xyxy[i].tolist()
            biou, bj = 0.0, -1
            for j, gb in enumerate(gt_boxes):
                if matched[j]:
                    continue
                v = iou(pb, gb)
                if v > biou:
                    biou, bj = v, j
            if biou >= 0.5:
                TP += 1
                matched[bj] = True
            else:
                FP += 1
        FN += sum(1 for m in matched if not m)

    precision = TP / (TP + FP) if TP + FP else 0.0
    recall = TP / (TP + FN) if TP + FN else 0.0
    print("\n===== FINAL REPORT CARD (test set: alag tournament ke clean frames) =====")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print("\nYe 'asli duniya ka andaza' hai — ye frames training me kabhi use nahi hue.")
    print("Valid-set wale numbers 'training ke dauraan' the; YE final report card hai.")


## Step 6 — ONNX export 📦

AEGIS ka backend model ko **ONNX format** me chalata hai, isliye trained model ko ONNX me badalna zaroori hai.

**Ek imaandaar note:** RF-DETR ka ONNX export *raw object queries* deta hai (har query ke liye box + class scores) — is par threshold lagana aur boxes ko scale karna backend ka kaam hai. Agar aapko seedha plug-and-play ONNX chahiye to **dusra notebook (`train_yolo26_nano_killfeed.ipynb`)** chalao — uska ONNX seedha `VISION_MODEL_PATH` me lagta hai. Dono me se jo behtar result de, wahi istemal karo.


In [ ]:
# Cell 8 — Best checkpoint → ONNX export
import glob, shutil, os

onnx_files = glob.glob("/kaggle/working/rfdetr_onnx/**/*.onnx", recursive=True)
if not onnx_files:
    # pehli baar: export karo (rfdetr[onnxexport] Cell 1 me install ho chuka hai)
    from rfdetr import RFDETRNano
    OUT = Path("/kaggle/working/rfdetr_output")
    ckpts = sorted(OUT.glob("*.pth"))
    best = [c for c in ckpts if "best" in c.name.lower()]
    best_ckpt = sorted(best)[-1] if best else ckpts[-1]
    model = RFDETRNano(pretrain_weights=str(best_ckpt))
    model.export(output_dir="/kaggle/working/rfdetr_onnx")
    onnx_files = glob.glob("/kaggle/working/rfdetr_onnx/**/*.onnx", recursive=True)

print("Exported ONNX:", onnx_files)
shutil.copy(onnx_files[0], "/kaggle/working/best.onnx")
size_mb = os.path.getsize("/kaggle/working/best.onnx") / 1e6
print(f"✔ /kaggle/working/best.onnx taiyaar ({size_mb:.1f} MB)")


## ✅ Ho gaya! Ab `best.onnx` download karo ⬇️

1. Kaggle page ke right side me **Output** section kholo (ya upar *File* → *Download*).
2. `best.onnx` download karo apne computer par.
3. Aage kya karna hai — ye `docs/KAGGLE_TRAINING.md` me likha hai: server par `VISION_MODEL_PATH=/path/to/best.onnx` set karo aur backend restart karo.

**Yaad rakho:** Model utna hi achha hoga jitna aapka dataset. Galat detections dikhen to dataset me aur alag-alag tarah ke frames (alag maps, alag kill-feed styles) jodho aur dobara train karo.


In [ ]:
# Cell 9 — Final check: best.onnx maujood hai?
import os
p = "/kaggle/working/best.onnx"
assert os.path.exists(p), "best.onnx nahi mila — Cell 8 dobara chalao!"
print(f"✔ {p}  ({os.path.getsize(p)/1e6:.1f} MB)")
print("Ab right side Output panel se download karo ⬇️")
